# Task 1.3: What the Paper Claims to Improve Over

**Paper**: Weisfeiler-Lehman Graph Kernels (Shervashidze et al., JMLR 2011)

**Student**: Meghavi (Roll: 230044)

## Baseline Method: The Ramon-Gärtner Subtree Kernel

**Reference**: Ramon, J. and Gärtner, T. (2003). Expressivity versus efficiency of graph kernels. *Proceedings of the First International Workshop on Mining Graphs, Trees and Sequences*, pp. 65–74.

The Ramon-Gärtner subtree kernel compares graphs by iteratively matching **all pairs of node neighborhoods**. For each pair of nodes $(v, w)$ from two graphs, it computes a kernel value by considering all possible matchings between subsets of their neighborhoods. The kernel is defined recursively:

$$k_h(v, w) = \lambda(l(v), l(w)) \cdot \sum_{R \subseteq M} \prod_{(a,b) \in R} k_{h-1}(a, b)$$

where $M$ ranges over all possible matchings between subsets of neighbors of $v$ and $w$. (Equation 3 in the WL paper, Section 3.2.2)

## Limitation Identified by the WL Paper

The Ramon-Gärtner kernel considers **ALL matchings of subsets** of node neighborhoods (Equation 3 in the paper). This makes its complexity **exponential in the node degree** $d$:

$$O(N^2 \cdot n^2 \cdot h \cdot 4^d)$$

where $N$ is the number of graphs, $n$ is the maximum number of nodes, $h$ is the subtree depth, and $d$ is the maximum node degree.

This exponential dependence on degree $d$ makes the Ramon-Gärtner kernel **impractical for large graphs** or graphs with high node degrees (e.g., social networks where nodes can have hundreds of connections). The paper explicitly states this limitation in Section 1.1 (last paragraph) and Section 3.2.2.

Additionally, the paper notes that computing the Ramon-Gärtner kernel on MUTAG (average degree ~2) already takes orders of magnitude longer than the WL kernel, and it becomes infeasible on larger benchmark datasets.

## How the WL Subtree Kernel Overcomes This Limitation

Instead of comparing **all possible matchings** between node neighborhoods, the WL subtree kernel checks for **exact neighborhood matches** via label compression (hashing). The key insight is:

1. **Aggregate** neighbor labels into a sorted string (deterministic, no combinatorial matching)
2. **Compress** via hashing (O(1) per node per iteration)
3. **Count** label occurrences across all iterations (histogram construction)
4. **Compare** via dot product of histograms

This reduces the per-iteration cost to **$O(m)$** — linear in the number of edges — regardless of node degree (Theorem 5, Section 3.2). The total kernel computation for $N$ graphs over $h$ iterations is $O(Nhm + N^2 h n)$ (Theorem 7, Section 3.2.1).

The WL kernel trades off the flexibility of partial neighborhood matching (which the Ramon-Gärtner kernel captures) for dramatic computational efficiency. It requires **exact** multiset matches rather than considering all partial matchings.

## Scenario Where the WL Subtree Kernel Would NOT Outperform

The WL subtree kernel's requirement for **exact neighborhood matches** can be a disadvantage when:

- Graphs are **small** (n < 30 nodes)
- Graphs have **many distinct labels** (rich label alphabet)
- Graphs have **low degree** (making the $4^d$ factor manageable)
- **Partial neighborhood similarity** matters for classification (i.e., two nodes with partially overlapping — but not identical — neighborhoods should be considered similar)

In this regime, the Ramon-Gärtner kernel's consideration of all subset matchings can capture **finer-grained structural similarity** that the WL kernel's exact-match approach misses.

**Empirical evidence from the paper**: Table 1 shows that on the **MUTAG** dataset (average 17.93 nodes, low degree), the Ramon-Gärtner subtree kernel achieves **85.72% accuracy** vs. the WL subtree kernel's **82.05%**. This is one of the few cases where a baseline outperforms WL, precisely because the graphs are small enough that the exponential complexity is acceptable, and partial matching provides useful signal.

More broadly, whenever the "soft" comparison of neighborhoods (partial matching, edit-distance-based matching) captures task-relevant structural variation that exact matching misses, the WL kernel will underperform despite its computational advantage.